# Convert between Transverse Mercator and WGS84

The Python code was rewritten from the C++ [PROJ](https://github.com/OSGeo/PROJ/tree/master) library.

## Structure implementations

In [1]:
from __future__ import annotations
from enum import Enum, IntEnum
import math
from pyproj import Transformer
import sys
from typing import overload, Optional

# Carto object implementations
class Coord:
    def __init__(self: Coord,
                 hemisphere: Hemisphere,
                 x: float,
                 y: float,
                 z: float,
                 zone: int) -> None:
        self.hemisphere = hemisphere
        self.x = x
        self.y = y
        self.z = z
        self.zone = zone

    def __repr__(self: Coord) -> str:
        return f'Coord({self.x}, {self.y}, {self.z}, hemisphere={self.hemisphere}, zone={self.zone})'

class EllipsoidDefinition:
    def __init__(self: EllipsoidDefinition,
                 semi_major: float,
                 inverse_flattening: float) -> None:
        self.a = semi_major
        self.rf = inverse_flattening
        self.f = 1 / self.rf
        self.b = self.a * (1 - self.f)
        self.e1_square = self.f * (2 - self.f)  # First eccentricity

    @property
    def e2_square(self: EllipsoidDefinition) -> float:
        '''
        The second eccentricity squared.
        '''
        return self.e1_square / (1 - self.e1_square)
    
    @property
    def e3_square(self: EllipsoidDefinition) -> float:
        '''
        The third eccentricity squared.
        '''
        return self.e1_square / (2 - self.e1_square)
    
    @property
    def f2(self: EllipsoidDefinition) -> float:
        '''
        The second flattening.
        '''
        return (self.a - self.b) / self.b
    
    @property
    def f3(self: EllipsoidDefinition) -> float:
        '''
        The third flattening.
        '''
        return (self.a - self.b) / (self.a + self.b)

class HelmertTransform:
    x = 0
    y = 0
    z = 0
    rx = 0
    ry = 0
    rz = 0
    s = 0
    len = 0
    
    def __init__(self: HelmertTransform,
                 params: list) -> None:
        if len(params) == 0:
            return
        elif len(params) == 3:
            self.x = params[0]
            self.y = params[1]
            self.z = params[2]
            self.len = 3
        elif len(params) == 7:
            self.x = params[0]
            self.y = params[1]
            self.z = params[2]
            self.rx = params[3]
            self.ry = params[4]
            self.rz = params[5]
            self.s = params[6]
            self.len = 7
        else:
            raise ValueError('The length should be 0, 3 or 7.')
        
    def __len__(self: HelmertTransform) -> int:
        return self.len

class Hemisphere(Enum):
    NORTH, SOUTH = range(2)

class ProjectionDefinition:
    def __init__(self: ProjectionDefinition,
                 ellipsoid: EllipsoidDefinition,
                 origin: double2,
                 shift: double2,
                 scale_factor: float,
                 transform: HelmertTransform) -> None:
        self.ellipsoid = ellipsoid
        self.origin = origin
        self.shift = shift
        self.scale_factor = scale_factor
        self.transform = transform

# Unity structure implementation
class double2:
    def __init__(self: double2,
                 x: float,
                 y: float) -> None:
        self.x = float(x)
        self.y = float(y)

    def __add__(self: double2,
                other: double2) -> double2:
        return double2(self.x + other.x, self.y + other.y)
    
    def __sub__(self: double2,
                other: double2) -> double2:
        return double2(self.x - other.x, self.y - other.y)
    
    def __mul__(self: double2,
                other: double2) -> double2:
        return double2(self.x * other.x, self.y * other.y)
    
    def __truediv__(self: double2,
                    other: double2) -> double2:
        return double2(self.x / other.x, self.y / other.y)
    
    def __pos__(self: double2) -> double2:
        return double2(self.x, self.y)

    def __neg__(self: double2) -> double2:
        return double2(-self.x, -self.y)
    
    def __repr__(self: double2) -> str:
        return f'double2({self.x}, {self.y})'

## DatumUtils class

In [2]:
# Implemented structures:
# [Carto.Geodata] Coord, EllipsoidDefinition, HelmertTransform, Hemisphere & ProjectionDefinition
# [Unity.Mathematics] double2

class DatumUtils(object):

    class AuxLat(IntEnum):
        '''
        The equivalent of `proj_internal.h - AuxLat`. The types of the auxiliary latitude.
        '''

        GEOGRAPHIC = 0,
        '''
        The geographic latitude (φ).
        '''

        PARAMETRIC = 1,
        '''
        The parametric latitude (β).
        '''

        GEOCENTRIC = 2,
        '''
        The geocentric latitude (θ).
        '''
        
        RECTIFYING = 3,
        '''
        The rectifying latitude (μ).
        '''
        
        CONFORMAL  = 4,
        '''
        The conformal latitude (χ).
        '''
        
        AUTHALIC   = 5,
        '''
        The authalic latitude (ξ).
        '''
        
        NUMBER     = 6
        '''
        The number of types of auxiliary latitudes.
        '''

    AUXILIARY_LATITUDE_POINTERS = [
        0, 0, 0, 0, 12, 33, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54,
        66, 66, 66, 66, 87, 87, 108, 108, 108, 129, 129, 129, 150, 150, 150,
        150, 150, 150
    ]

    # PROJ contains the following transformation by default:
    #   phi <-> mu for meridian distance
    #   phi <-> chi for tmerc
    #   phi <-> xi for authalic latitude conversions
    #   chi <-> mu for tmerc
    AUXILIARY_LATITUDE_SERIES = [
        # C[phi,phi] skipped
        # C[phi,beta] skipped
        # C[phi,theta] skipped
        # C[phi,mu]; even coeffs only
        3.0/2.0, -27.0/32.0, 269.0/512.0,
        21.0/16.0, -55.0/32.0, 6759.0/4096.0,
        151.0/96.0, -417.0/128.0,
        1097.0/512.0, -15543.0/2560.0,
        8011.0/2560.0,
        293393.0/61440.0,
        # C[phi,chi]
        2.0, -2.0/3.0, -2.0, 116.0/45.0, 26.0/45.0, -2854.0/675.0,
        7.0/3.0, -8.0/5.0, -227.0/45.0, 2704.0/315.0, 2323.0/945.0,
        56.0/15.0, -136.0/35.0, -1262.0/105.0, 73814.0/2835.0,
        4279.0/630.0, -332.0/35.0, -399572.0/14175.0,
        4174.0/315.0, -144838.0/6237.0,
        601676.0/22275.0,
        # C[phi,xi]
        4.0/3.0, 4.0/45.0, -16.0/35.0, -2582.0/14175.0, 60136.0/467775.0,
        28112932.0/212837625.0,
        46.0/45.0, 152.0/945.0, -11966.0/14175.0, -21016.0/51975.0,
        251310128.0/638512875.0,
        3044.0/2835.0, 3802.0/14175.0, -94388.0/66825.0, -8797648.0/10945935.0,
        6059.0/4725.0, 41072.0/93555.0, -1472637812.0/638512875.0,
        768272.0/467775.0, 455935736.0/638512875.0,
        4210684958.0/1915538625.0,
        # C[beta,phi] skipped
        # C[beta,beta] skipped
        # C[beta,theta] skipped
        # C[beta,mu] skipped
        # C[beta,chi] skipped
        # C[beta,xi] skipped
        # C[theta,phi] skipped
        # C[theta,beta] skipped
        # C[theta,theta] skipped
        # C[theta,mu] skipped
        # C[theta,chi] skipped
        # C[theta,xi] skipped
        # C[mu,phi]; even coeffs only
        -3.0/2.0, 9.0/16.0, -3.0/32.0,
        15.0/16.0, -15.0/32.0, 135.0/2048.0,
        -35.0/48.0, 105.0/256.0,
        315.0/512.0, -189.0/512.0,
        -693.0/1280.0,
        1001.0/2048.0,
        # C[mu,beta] skipped
        # C[mu,theta] skipped
        # C[mu,mu] skipped
        # C[mu,chi]
        1.0/2.0, -2.0/3.0, 5.0/16.0, 41.0/180.0, -127.0/288.0, 7891.0/37800.0,
        13.0/48.0, -3.0/5.0, 557.0/1440.0, 281.0/630.0, -1983433.0/1935360.0,
        61.0/240.0, -103.0/140.0, 15061.0/26880.0, 167603.0/181440.0,
        49561.0/161280.0, -179.0/168.0, 6601661.0/7257600.0,
        34729.0/80640.0, -3418889.0/1995840.0,
        212378941.0/319334400.0,
        # C[mu,xi] skipped
        # C[chi,phi]
        -2.0, 2.0/3.0, 4.0/3.0, -82.0/45.0, 32.0/45.0, 4642.0/4725.0,
        5.0/3.0, -16.0/15.0, -13.0/9.0, 904.0/315.0, -1522.0/945.0,
        -26.0/15.0, 34.0/21.0, 8.0/5.0, -12686.0/2835.0,
        1237.0/630.0, -12.0/5.0, -24832.0/14175.0,
        -734.0/315.0, 109598.0/31185.0,
        444337.0/155925.0,
        # C[chi,beta] skipped
        # C[chi,theta] skipped
        # C[chi,mu]
        -1.0/2.0, 2.0/3.0, -37.0/96.0, 1.0/360.0, 81.0/512.0,
        -96199.0/604800.0,
        -1.0/48.0, -1.0/15.0, 437.0/1440.0, -46.0/105.0, 1118711.0/3870720.0,
        -17.0/480.0, 37.0/840.0, 209.0/4480.0, -5569.0/90720.0,
        -4397.0/161280.0, 11.0/504.0, 830251.0/7257600.0,
        -4583.0/161280.0, 108847.0/3991680.0,
        -20648693.0/638668800.0,
        # C[chi,chi] skipped
        # C[chi,xi] skipped
        # C[xi,phi]
        -4.0/3.0, -4.0/45.0, 88.0/315.0, 538.0/4725.0, 20824.0/467775.0,
        -44732.0/2837835.0,
        34.0/45.0, 8.0/105.0, -2482.0/14175.0, -37192.0/467775.0,
        -12467764.0/212837625.0,
        -1532.0/2835.0, -898.0/14175.0, 54968.0/467775.0,
        100320856.0/1915538625.0,
        6007.0/14175.0, 24496.0/467775.0, -5884124.0/70945875.0,
        -23356.0/66825.0, -839792.0/19348875.0,
        570284222.0/1915538625.0
        # C[xi,beta] skipped
        # C[xi,theta] skipped
        # C[xi,mu] skipped
        # C[xi,chi] skipped
        # C[xi,xi] skipped
    ]

    @staticmethod
    def auxiliary_latitude_coefficients(n: float,
                                        coeffs: list[float],
                                        pointers: list[int],
                                        aux_in: AuxLat,
                                        aux_out: AuxLat) -> list[float]:
        '''
        The equivalent of `latitudes.cpp - pj_auxlat_coeffs()`. The output would always be an array with length of 6.
        '''
        if not (0 <= aux_in < DatumUtils.AuxLat.NUMBER and 0 <= aux_out < DatumUtils.AuxLat.NUMBER):
            raise ValueError('Invalid auxiliary latitude specifier.')
        
        k = DatumUtils.AuxLat.NUMBER * aux_out + aux_in
        o = pointers[k]
        if o == pointers[k + 1]:
            raise ValueError("Unsupported conversion between auxiliary latitudes.")
        
        f: list[float] = []
        l_max = 6
        d = n
        n2 = n * n

        if aux_in <= DatumUtils.AuxLat.RECTIFYING and aux_out <= DatumUtils.AuxLat.RECTIFYING:
            for l in range(l_max):
                m = (l_max - l - 1) // 2
                val = d * DatumUtils.polynomial_sum(n2, coeffs[o:o + m + 1], m)
                f.append(val)
                o += m + 1
                d *= n
        else:
            for l in range(l_max):
                m = l_max - l - 1
                val = d * DatumUtils.polynomial_sum(n, coeffs[o:o + m + 1], m)
                f.append(val)
                o += m + 1
                d *= n

        return f

    @staticmethod
    def clenshaw_summation(a: list[float],
                           sin_arg_r: float,
                           cos_arg_r: float,
                           sinh_arg_i: float,
                           cosh_arg_i: float) -> tuple[float, float]:
        '''
        The equivalent of `tmerc.cpp - clenS().` Note that this function returns `tuple[float, float]`, not original's `double`.
        '''
        size = len(a)
        r = 2 * cos_arg_r * cosh_arg_i
        i = -2 * sin_arg_r * sinh_arg_i

        hi1 = hr1 = hi = 0.0
        p = size - 1
        hr = a[p]
        p -= 1

        while p >= 0:
            hr2 = hr1
            hi2 = hi1
            hr1 = hr
            hi1 = hi
            hr = -hr2 + r * hr1 - i * hi1 + a[p]
            hi = -hi2 + i * hr1 + r * hi1
            p -= 1

        r_final = sin_arg_r * cosh_arg_i
        i_final = cos_arg_r * sinh_arg_i
        R = r_final * hr - i_final * hi
        I = r_final * hi + i_final * hr
        return R, I

    @overload
    def convert_auxiliary_latitude(zeta: float,
                                   coeffs: list[float]) -> float:
        '''
        The equivalent of `latitudes.cpp - pj_auxlat_convert()`.
        '''
        ...
    
    @overload
    def convert_auxiliary_latitude(zeta: float,
                                   coeffs: list[float],
                                   s_zeta: float,
                                   c_zeta: float) -> float:
        '''
        The equivalent of `latitudes.cpp - pj_auxlat_convert()`.
        '''
        ...

    @staticmethod
    def convert_auxiliary_latitude(zeta: float,
                                   coeffs: list[float],
                                   s_zeta: Optional[float] = None,
                                   c_zeta: Optional[float] = None) -> float:
        '''
        The equivalent of `latitudes.cpp - pj_auxlat_convert()`.
        '''
        if (s_zeta is None) | (c_zeta is None):
            return DatumUtils.convert_auxiliary_latitude(zeta, coeffs, math.sin(zeta), math.cos(zeta))
        else:
            return zeta + DatumUtils.evaluate_clenshaw(coeffs, s_zeta, c_zeta)
        
    @staticmethod
    def evaluate_clenshaw(coeffs: list[float],
                          s_zeta: float,
                          c_zeta: float) -> float:
        '''
        The equivalent of `latitudes.cpp - pj_clenshaw().` This method utilize Clenshaw summation to calculate sum(coeffs[k] * sin((2*k+2) * zeta), k, 0, 5). 
        '''
        order = 6
        u0 = u1 = 0
        x = 2 * (c_zeta - s_zeta) * (c_zeta + s_zeta)

        while (order > 0):
            order -= 1
            t = x * u0 - u1 + coeffs[order]
            u1 = u0
            u0 = t
        
        return 2 * s_zeta * c_zeta * u0

    @staticmethod
    def evaluate_rectifying_radius(n: float) -> float:
        '''
        The equivalent of `latitudes.cpp - pj_rectifying_radius()`.
        '''
        r: list[float] = [1., 1.0 / 4, 1.0 / 64, 1.0 / 256]
        return DatumUtils.polynomial_sum(n * n, r, 3) / (n + 1)

    @staticmethod
    def polynomial_sum(x: float,
                       coeffs: list[float],
                       order: int) -> float:
        '''
        The equivalent of `latitudes.cpp - pj_polyval().` This method utilize [Horner's method](https://en.wikipedia.org/wiki/Horner%27s_method) to calculate sum(coeffs[i] * x^i, i, 0, order).
        '''
        y: float = 0. if order < 0 else coeffs[order]
        while order > 0:
            order -= 1
            y = y * x + coeffs[order]
        return y

## Transform class

In [3]:
class Transform(object):
    @staticmethod
    def transverse_mercator_to_wgs84(coord: Coord,
                                     projection: ProjectionDefinition) -> Coord:
        '''
        The equivalent of `tmerc.cpp - exact_e_inv()`.
        '''
        
        result = Coord(Hemisphere.NORTH, x=0, y=0, z=0, zone=0)

        lat0 = projection.origin.y / 180 * math.pi
        lon0 = projection.origin.x / 180 * math.pi
        x = (coord.x - projection.shift.x) / projection.ellipsoid.a
        y = (coord.y - projection.shift.y) / projection.ellipsoid.a

        n = projection.ellipsoid.f3
        '''
        The third flattening.
        '''

        meridian_quad = projection.scale_factor * DatumUtils.evaluate_rectifying_radius(n)
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.Qn`. Merid. quad., scaled to the projection.
        '''

        conformal_to_geographic = DatumUtils.auxiliary_latitude_coefficients(
            n,
            DatumUtils.AUXILIARY_LATITUDE_SERIES,
            DatumUtils.AUXILIARY_LATITUDE_POINTERS,
            DatumUtils.AuxLat.CONFORMAL,
            DatumUtils.AuxLat.GEOGRAPHIC
        )
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.cgb`
        '''

        conformal_to_rectifying = DatumUtils.auxiliary_latitude_coefficients(
            n,
            DatumUtils.AUXILIARY_LATITUDE_SERIES,
            DatumUtils.AUXILIARY_LATITUDE_POINTERS,
            DatumUtils.AuxLat.CONFORMAL,
            DatumUtils.AuxLat.RECTIFYING
        )
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.gtu`.
        '''

        geographic_to_conformal = DatumUtils.auxiliary_latitude_coefficients(
            n,
            DatumUtils.AUXILIARY_LATITUDE_SERIES,
            DatumUtils.AUXILIARY_LATITUDE_POINTERS,
            DatumUtils.AuxLat.GEOGRAPHIC,
            DatumUtils.AuxLat.CONFORMAL
        )
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.cbg`
        '''

        rectifying_to_conformal = DatumUtils.auxiliary_latitude_coefficients(
            n,
            DatumUtils.AUXILIARY_LATITUDE_SERIES,
            DatumUtils.AUXILIARY_LATITUDE_POINTERS,
            DatumUtils.AuxLat.RECTIFYING,
            DatumUtils.AuxLat.CONFORMAL
        )
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.utg`.
        '''

        gaussian_origin_latitude = DatumUtils.convert_auxiliary_latitude(lat0, geographic_to_conformal)
        '''
        Gaussian latitude at origin.
        '''

        radius_vector = -meridian_quad * DatumUtils.convert_auxiliary_latitude(gaussian_origin_latitude, conformal_to_rectifying)
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.Zb`. Radius vector in polar coord. systems.
        '''

        Ce = x / meridian_quad
        Cn = (y - radius_vector) / meridian_quad

        if (abs(Ce) <= 2.623395162778):
            sin_arg_r = math.sin(2 * Cn)
            cos_arg_r = math.cos(2 * Cn)
            exp_2_Ce = math.exp(2 * Ce)
            half_inv_exp_2_Ce = 0.5 / exp_2_Ce
            sinh_arg_i = 0.5 * exp_2_Ce - half_inv_exp_2_Ce
            cosh_arg_i = 0.5 * exp_2_Ce + half_inv_exp_2_Ce
            r, i = DatumUtils.clenshaw_summation(
                rectifying_to_conformal,
                sin_arg_r,
                cos_arg_r,
                sinh_arg_i,
                cosh_arg_i
            )
            Ce += i
            Cn += r
            sin_Cn = math.sin(Cn)
            cos_Cn = math.cos(Cn)
            sinh_Ce = math.sinh(Ce)
            Ce = math.atan2(sinh_Ce, cos_Cn)
            modulus_Ce = math.hypot(sinh_Ce, cos_Cn)
            rr = math.hypot(sin_Cn, modulus_Ce)
            Cn = math.atan2(sin_Cn, modulus_Ce)

            result.x = (Ce + lon0) * 180 / math.pi
            result.y = DatumUtils.convert_auxiliary_latitude(
                Cn,
                conformal_to_geographic,
                sin_Cn / rr,
                modulus_Ce / rr
            ) * 180 / math.pi
            if (result.y < 0):
                result.hemisphere = Hemisphere.SOUTH

        else:
            result.x = sys.float_info.max
            result.y = sys.float_info.max

        return result
    
    @staticmethod
    def wgs84_to_transverse_mercator(coord: Coord,
                                     projection: ProjectionDefinition) -> Coord:
        '''
        The equivalent of `tmerc.cpp - exact_e_fwd()`.
        '''

        result = Coord(Hemisphere.NORTH, x=0, y=0, z=0, zone=0)

        lat = coord.y / 180 * math.pi
        lat0 = projection.origin.y / 180 * math.pi
        lon = coord.x / 180 * math.pi
        lon0 = projection.origin.x / 180 * math.pi

        n = projection.ellipsoid.f3
        '''
        The third flattening.
        '''

        meridian_quad = projection.scale_factor * DatumUtils.evaluate_rectifying_radius(n)
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.Qn`. Merid. quad., scaled to the projection.
        '''

        conformal_to_rectifying = DatumUtils.auxiliary_latitude_coefficients(
            n,
            DatumUtils.AUXILIARY_LATITUDE_SERIES,
            DatumUtils.AUXILIARY_LATITUDE_POINTERS,
            DatumUtils.AuxLat.CONFORMAL,
            DatumUtils.AuxLat.RECTIFYING
        )
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.gtu`.
        '''

        geographic_to_conformal = DatumUtils.auxiliary_latitude_coefficients(
            n,
            DatumUtils.AUXILIARY_LATITUDE_SERIES,
            DatumUtils.AUXILIARY_LATITUDE_POINTERS,
            DatumUtils.AuxLat.GEOGRAPHIC,
            DatumUtils.AuxLat.CONFORMAL
        )
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.cbg`
        '''
        
        gaussian_origin_latitude = DatumUtils.convert_auxiliary_latitude(lat0, geographic_to_conformal)
        '''
        Gaussian latitude at origin.
        '''

        radius_vector = -meridian_quad * DatumUtils.convert_auxiliary_latitude(gaussian_origin_latitude, conformal_to_rectifying)
        '''
        The equivalent of `tmerc.cpp - PoderEngsager.Zb`. Radius vector in polar coord. systems.
        '''

        Ce = lon - lon0
        Cn = DatumUtils.convert_auxiliary_latitude(lat, geographic_to_conformal)
        sin_Cn = math.sin(Cn)
        cos_Cn = math.cos(Cn)
        sin_Ce = math.sin(Ce)
        cos_Ce = math.cos(Ce)
        cos_Cn_cos_Ce = cos_Cn * cos_Ce
        Cn = math.atan2(sin_Cn, cos_Cn_cos_Ce)
        inv_denom_tan_Ce = 1. / math.hypot(sin_Cn, cos_Cn_cos_Ce)
        inv_denom_tan_Ce_squared = inv_denom_tan_Ce * inv_denom_tan_Ce
        tan_Ce = sin_Ce * cos_Cn * inv_denom_tan_Ce
        Ce = math.asinh(tan_Ce)
        tmp_r = 2 * cos_Cn_cos_Ce * inv_denom_tan_Ce_squared
        sin_arg_r = sin_Cn * tmp_r
        cos_arg_r = cos_Cn_cos_Ce * tmp_r - 1
        sinh_arg_i = 2 * tan_Ce * inv_denom_tan_Ce
        cosh_arg_i = 2 * inv_denom_tan_Ce_squared - 1
        r, i = DatumUtils.clenshaw_summation(
            conformal_to_rectifying,
            sin_arg_r,
            cos_arg_r,
            sinh_arg_i,
            cosh_arg_i
        )
        Ce += i
        Cn += r

        if (abs(Ce) <= 2.623395162778):
            result.x = meridian_quad * Ce * projection.ellipsoid.a + projection.shift.x
            result.y = (meridian_quad * Cn + radius_vector) * projection.ellipsoid.a + projection.shift.y
        else:
            result.x = sys.float_info.max
            result.y = sys.float_info.max
        
        return result

## Test the methods

In [4]:
# TWD97/TM2 coord
coord1 = Coord(Hemisphere.NORTH, 304563.458, 2769640.183, 0., 0.)

# WGS84 coord
coord2 = Coord(Hemisphere.NORTH, 121.540697, 25.033890, 0., 0.)

# GRS80 Ellipsoid
grs80 = EllipsoidDefinition(6378137, 298.257222101)

# WGS84 Ellipsoid
wgs84 = EllipsoidDefinition(6378137, 298.257223563)

# TWD97/TM2 projection
twd97 = ProjectionDefinition(grs80, double2(121., 0.), double2(250000., 0.), 0.9999, HelmertTransform([]))

# PYPROJ result
fwd = Transformer.from_crs("EPSG:4326", "EPSG:3826", always_xy=True)
inv = Transformer.from_crs("EPSG:3826", "EPSG:4326", always_xy=True)
print('PYPROJ result')
display(inv.transform(coord1.x, coord1.y))
display(fwd.transform(coord2.x, coord2.y))

# Carto implementation
print('Carto Implementation')
display(Transform.transverse_mercator_to_wgs84(coord1, twd97))
display(Transform.wgs84_to_transverse_mercator(coord2, twd97))

PYPROJ result


(121.54069662192991, 25.03388952477141)

(304563.49594280886, 2769640.2357912934)

Carto Implementation


Coord(121.54069662192991, 25.033889524771407, 0, hemisphere=Hemisphere.NORTH, zone=0)

Coord(304563.4959428063, 2769640.2357912934, 0, hemisphere=Hemisphere.NORTH, zone=0)